In [ ]:
"""
Pull World Cup 2026 squad and player stats from API-Football, based on
the verified documentation structure (not guessed).
 
Key facts confirmed from the docs:
  - /players/squads: pass a team ID, no season needed, returns the full
    roster (name, age, number, position, photo) in ONE unpaginated call.
    This is what we use for "who is on this team."
  - /players: requires team+season (or id+season, or league+season) and
    PAGINATES AT 20 RESULTS PER PAGE. You must loop through paging.total.
    Stats are grouped per competition the player appeared in -- sum
    across blocks yourself if you want a season total.
  - Team IDs are stable across seasons/competitions -- look them up once
    via /teams and reuse.
"""

In [3]:
import requests
import pandas as pd
import time
import os

In [ ]:
#Variables
API_KEY = "masked"
BASE_URL = "https://v3.football.api-sports.io"
HEADERS = {"x-apisports-key": API_KEY}
DELAY_BETWEEN_CALLS = 0.3

SEASONS = (2023, 2024, 2025)
 
QUALIFIED_TEAMS_2026 = [
    "Canada", "Brazil", "Paraguay", "Morocco", "Norway", "France", "Mexico",
    "England", "Belgium", "United States", "Spain", "Portugal", "Switzerland",
    "Egypt", "Argentina", "Colombia",
    "South Africa", "Japan", "Germany", "Netherlands", "Ivory Coast", "Sweden",
    "Ecuador", "DR Congo", "Senegal", "Bosnia and Herzegovina", "Austria",
    "Croatia", "Algeria", "Australia", "Cape Verde", "Ghana",
    "Czech Republic", "South Korea", "Qatar", "Scotland", "Haiti", "Turkey",
    "Curaçao", "Tunisia", "New Zealand", "Iran", "Saudi Arabia", "Uruguay",
    "Iraq", "Jordan", "Uzbekistan", "Panama",
]

assert len(QUALIFIED_TEAMS_2026) == 48

In [8]:
def api_get(endpoint: str, params: dict, max_retries: int = 5) -> dict:
    for attempt in range(max_retries):
        resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params)
        remaining_day = resp.headers.get("x-ratelimit-requests-remaining")
        if remaining_day is not None and int(remaining_day) <= 20:
            print(f"  WARNING: only {remaining_day} daily requests left.")
        if resp.status_code == 429:
            retry_after = int(resp.headers.get("Retry-After", 0))
            wait_time = retry_after if retry_after > 0 else (2 ** attempt) * 5
            print(f"  Rate limited. Waiting {wait_time}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait_time)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception("Still rate-limited after max retries.")

# =========================================================================
# STAGE A: Resolve and verify all 48 team IDs before spending the big
# batch of player-stat calls. National teams in this API are typically
# tagged as their own "team" entity distinct from club teams of the same
# name -- verify by checking the country/national fields in the output.
# =========================================================================
def resolve_all_team_ids(nations=QUALIFIED_TEAMS_2026) -> pd.DataFrame:
    rows = []
    for nation in nations:
        data = api_get("teams", {"name": nation})["response"]
        if not data:
            rows.append({"nation": nation, "team_id": None, "matched_name": None,
                         "country": None, "national": None, "status": "NOT FOUND"})
            print(f"  [MISS] {nation}")
        else:
            # Prefer an entry explicitly flagged as a national team if present
            national_entries = [t for t in data if t["team"].get("national")]
            best = national_entries[0] if national_entries else data[0]
            rows.append({
                "nation": nation,
                "team_id": best["team"]["id"],
                "matched_name": best["team"]["name"],
                "country": best["team"]["country"],
                "national": best["team"].get("national"),
                "status": "OK" if national_entries else "CHECK -- no 'national' flag found",
            })
            flag = "OK" if national_entries else "CHECK"
            print(f"  [{flag}] {nation} -> id={best['team']['id']}  name={best['team']['name']}  national={best['team'].get('national')}")
        time.sleep(DELAY_BETWEEN_CALLS)
    return pd.DataFrame(rows)
 
 
# =========================================================================
# STAGE B: Raw, unaggregated player stats pull for every team/player/season.
# No aggregation -- one row per (player, season, competition) exactly as
# returned by the API. Saves incrementally per team.
# =========================================================================
def get_squad(team_id: int) -> pd.DataFrame:
    data = api_get("players/squads", {"team": team_id})["response"]
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data[0]["players"])
    df["team_id"] = team_id
    df["team_name"] = data[0]["team"]["name"]
    return df
 
 
def get_player_club_stats_raw(player_id: int, season: int) -> pd.DataFrame:
    data = api_get("players", {"id": player_id, "season": season})
    if data["errors"] or not data["response"]:
        return pd.DataFrame()
 
    rows = []
    profile = data["response"][0]["player"]
    for stat_block in data["response"][0]["statistics"]:
        games = stat_block.get("games", {}) or {}
        goals = stat_block.get("goals", {}) or {}
        passes = stat_block.get("passes", {}) or {}
        shots = stat_block.get("shots", {}) or {}
        tackles = stat_block.get("tackles", {}) or {}
        duels = stat_block.get("duels", {}) or {}
        rows.append({
            "player_id": profile["id"], "name": profile["name"], "age": profile["age"],
            "nationality": profile["nationality"], "season": season,
            "club": stat_block.get("team", {}).get("name"),
            "competition": stat_block.get("league", {}).get("name"),
            "country": stat_block.get("league", {}).get("country"),
            "appearances": games.get("appearences"), "minutes": games.get("minutes"),
            "position": games.get("position"), "rating": games.get("rating"),
            "goals": goals.get("total"), "assists": goals.get("assists"),
            "passes_total": passes.get("total"), "passes_key": passes.get("key"),
            "pass_accuracy_pct": passes.get("accuracy"),
            "shots_total": shots.get("total"), "shots_on_target": shots.get("on"),
            "tackles_total": tackles.get("total"),
            "duels_total": duels.get("total"), "duels_won": duels.get("won"),
        })
    return pd.DataFrame(rows)
 
 
def pull_all_teams_raw(team_ids_df: pd.DataFrame, seasons=SEASONS, out_dir="raw_pull"):
    os.makedirs(out_dir, exist_ok=True)
    master_path = os.path.join(out_dir, "all_players_raw_stats.csv")
    first_write = not os.path.exists(master_path)
 
    valid_teams = team_ids_df[team_ids_df["team_id"].notna()]
    for _, row in valid_teams.iterrows():
        nation, team_id = row["nation"], int(row["team_id"])
        print(f"\n--- {nation} (team_id={team_id}) ---")
 
        squad = get_squad(team_id)
        squad.to_csv(os.path.join(out_dir, f"squad_{nation.replace(' ', '_')}.csv"), index=False)
        time.sleep(DELAY_BETWEEN_CALLS)
 
        team_rows = []
        for _, player in squad.iterrows():
            for season in seasons:
                stats = get_player_club_stats_raw(player["id"], season)
                if not stats.empty:
                    stats["roster_nation"] = nation
                    team_rows.append(stats)
                time.sleep(DELAY_BETWEEN_CALLS)
 
        if team_rows:
            team_df = pd.concat(team_rows, ignore_index=True)
            team_df.to_csv(
                master_path, mode="a", header=first_write, index=False
            )
            first_write = False
            print(f"  Saved {len(team_df)} rows for {nation} -> appended to {master_path}")

In [18]:
if __name__ == "__main__":
    # =====================================================================
    # STEP 1: Pull the roster (squad membership) for ONE team -- Argentina.
    # This is the cheap, unpaginated call we already confirmed works.
    # Cost: 2 calls (find_team_id + get_squad).
    # =====================================================================
    print("=== Step 1: Pulling Argentina roster ===")
    team_id = find_team_id("Argentina")
    print(f"Argentina team_id = {team_id}")
 
    squad = get_squad(team_id)
    squad.to_csv("argentina_squad.csv", index=False)
    print(f"Squad ({len(squad)} players) -> saved to argentina_squad.csv")
    print(squad[["id", "name", "age", "position"]])
 
    # =====================================================================
    # STEP 2 (SCALED DOWN FOR FREE PLAN): while still on the free plan's
    # 100/day quota, only test on a SMALL sample -- 3 players, 1 season --
    # rather than the full 26 players x 3 seasons (78 calls). This costs
    # just 3 calls, leaving plenty of quota for debugging/re-runs today.
    # Once the Pro plan is active, bump sample_players back to the full
    # squad and seasons back to (2022, 2023, 2024).
    # =====================================================================
    print("\n=== Step 2 (SAMPLE TEST -- free plan quota safe): 3 players, 1 season ===")
    sample_players = squad.head(3)
    club_stats = get_full_roster_club_stats(sample_players, seasons=(2024,))
    club_stats.to_csv("argentina_player_club_stats_SAMPLE.csv", index=False)
    print(f"\nPulled stats for {club_stats['player_id'].nunique() if not club_stats.empty else 0} "
          f"of {len(sample_players)} sample players -> saved to argentina_player_club_stats_SAMPLE.csv")
    print(club_stats.head(15))
    print("\nOnce Pro plan is active, re-run with the full squad and seasons=(2022,2023,2024).")

=== Step 1: Pulling Argentina roster ===
Argentina team_id = 26
Squad (26 players) -> saved to argentina_squad.csv
        id               name  age    position
0    19599        E. Martínez   33  Goalkeeper
1     2465           J. Musso   31  Goalkeeper
2    47296           G. Rulli   33  Goalkeeper
3     2467  Lisandro Martínez   27    Defender
4     6231          F. Medina   26    Defender
5     6503          N. Molina   27    Defender
6     2468         G. Montiel   28    Defender
7      624        N. Otamendi   37    Defender
8    30776          C. Romero   27    Defender
9      529      N. Tagliafico   33    Defender
10  319572           V. Barco   21  Midfielder
11    5996       E. Fernández   24  Midfielder
12    1578        G. Lo Celso   29  Midfielder
13    6716    A. Mac Allister   27  Midfielder
14    6002        E. Palacios   27    Defender
15     271         L. Paredes   31  Midfielder
16   26315        N. González   27  Midfielder
17    2472         R. De Paul   31  Mid

In [ ]:
if __name__ == "__main__":
    print("=== STAGE A: Resolving and verifying all 48 team IDs ===")
    team_ids_df = resolve_all_team_ids()
    team_ids_df.to_csv("wc2026_team_ids.csv", index=False)
    print(f"\nSaved -> wc2026_team_ids.csv")
    print(f"Found: {team_ids_df['team_id'].notna().sum()} / 48")
    print(f"Flagged for manual check ('national' flag missing): "
          f"{(team_ids_df['status'] == 'CHECK -- no national flag found').sum()}")
    print("\n>>> STOP HERE. Review wc2026_team_ids.csv manually before running Stage B. <<<")
 
    # Uncomment only after verifying wc2026_team_ids.csv looks correct:
    # print("\n=== STAGE B: Pulling raw player stats for all 48 teams (2023-2025) ===")
    # pull_all_teams_raw(team_ids_df, seasons=SEASONS)

=== STAGE A: Resolving and verifying all 48 team IDs ===
  [OK] Canada -> id=5529  name=Canada  national=True
  [OK] Brazil -> id=6  name=Brazil  national=True
  [OK] Paraguay -> id=2380  name=Paraguay  national=True
  [OK] Morocco -> id=31  name=Morocco  national=True
  [OK] Norway -> id=1090  name=Norway  national=True
  [OK] France -> id=2  name=France  national=True
  [OK] Mexico -> id=16  name=Mexico  national=True
  [OK] England -> id=10  name=England  national=True
  [OK] Belgium -> id=1  name=Belgium  national=True
  [MISS] United States
  [OK] Spain -> id=9  name=Spain  national=True
  [OK] Portugal -> id=27  name=Portugal  national=True
  [OK] Switzerland -> id=15  name=Switzerland  national=True
  [OK] Egypt -> id=32  name=Egypt  national=True
  [OK] Argentina -> id=26  name=Argentina  national=True
  [OK] Colombia -> id=8  name=Colombia  national=True
  [OK] South Africa -> id=1531  name=South Africa  national=True
  [OK] Japan -> id=12  name=Japan  national=True
  [OK] Ger

: 